# 01 — Collision Schema Exploration

## Purpose and Scope

This notebook inspects the raw City of Ottawa Traffic Collision dataset **before** it is loaded into DuckDB.

Goals:
- Validate the schema (columns, dtypes, nulls)
- Audit categorical value distributions
- Quantify nulls in count columns
- Check coordinate quality
- Verify identifier uniqueness


## Imports and File Paths

In [1]:
import pandas as pd
from pathlib import Path

RAW_DATA_PATH = Path("../data/raw/Traffic_Collision_Data.csv")

if not RAW_DATA_PATH.exists():
    print(f"WARNING: File not found at {RAW_DATA_PATH.resolve()}")
else:
    print(f"File found: {RAW_DATA_PATH.resolve()}")

File found: C:\Users\johna\ottawa-road-safety-analytics\data\raw\Traffic_Collision_Data.csv


## Load Raw Dataset

In [2]:
df = pd.read_csv(RAW_DATA_PATH, low_memory=False)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")

Loaded 94,406 rows and 28 columns


## Dataset Shape and Column Overview

In [3]:
print("Shape:", df.shape)
print("\nColumns and dtypes:")
print(df.dtypes.to_string())

Shape: (94406, 28)

Columns and dtypes:
X                             float64
Y                             float64
X_Coordinate                  float64
Y_Coordinate                  float64
ID                                str
Geo_ID                            str
Accident_Year                   int64
Accident_Date                     str
Location                          str
Classification_Of_Accident        str
Initial_Impact_Type               str
Road_1_Surface_Condition          str
Environment_Condition_1           str
Light                             str
Traffic_Control                   str
num_of_vehicles               float64
num_of_pedestrians            float64
num_of_bicycles               float64
num_of_motorcycles            float64
Max_injury                        str
num_of_injuries               float64
num_of_minimal                float64
num_of_minor                  float64
num_of_major                  float64
num_of_fatal                  float64
Lat       

In [ ]:
df.head()

Sample (5 rows):


,X,Y,X_Coordinate,Y_Coordinate,ID,Geo_ID,Accident_Year,Accident_Date,Location,Classification_Of_Accident,...,num_of_motorcycles,Max_injury,num_of_injuries,num_of_minimal,num_of_minor,num_of_major,num_of_fatal,Lat,Long,ObjectId
0,-8.449907e+06,5.674412e+06,351293.8071,5021841.110,2017--101,__3Z00RNB,2017,1/4/2017,MARCH RD btwn 280 S OF CARLING AVE/STATION RD ...,02 - Non-fatal injury,...,NaN,Minimal,1.0,1.0,NaN,NaN,NaN,45.334978,-75.906802,1
1,-8.432341e+06,5.664915e+06,363723.6849,5015276.436,2017--201,0004726,2017,1/5/2017,GREENBANK RD @ BERRIGAN DR/WESSEX RD (0004726),03 - P.D. only,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.274975,-75.749006,2
2,-8.427552e+06,5.687484e+06,366942.7019,5031143.575,2017--801,0006357,2017,1/23/2017,ALBERT ST @ BAY ST (0006357),02 - Non-fatal injury,...,NaN,Minimal,1.0,1.0,NaN,NaN,NaN,45.417468,-75.705989,3
3,-8.423618e+06,5.682501e+06,369744.7181,5027678.543,2017--102,0001586,2017,1/4/2017,KILBORN AVE/KILBORN PL @ LAMIRA ST (0001586),03 - P.D. only,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.386036,-75.670647,4
4,-8.421028e+06,5.632923e+06,371935.0538,4992842.240,2017--103,__3Z09V2,2017,1/4/2017,DONNELLY DR btwn FAIRMILE RD & FOURTH LINE RD ...,03 - P.D. only,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.072377,-75.647379,5


## Column Summary Table

Full null profile for every column.

In [ ]:
# just checking null count
summary = pd.DataFrame({
    "column_name": df.columns,
    "dtype": df.dtypes.values,
    "non_null_count": df.notna().sum().values,
    "null_count": df.isna().sum().values,
    "null_pct": (df.isna().sum().values / len(df) * 100).round(2),
    "unique_count": df.nunique().values,
})

pd.set_option("display.max_rows", None)
summary

,column_name,dtype,non_null_count,null_count,null_pct,unique_count
0,X,float64,94406,0,0.00,57587
1,Y,float64,94406,0,0.00,57587
2,X_Coordinate,float64,94406,0,0.00,57245
3,Y_Coordinate,float64,94406,0,0.00,54946
4,ID,str,94406,0,0.00,94406
5,Geo_ID,str,92330,2076,2.20,14900
6,Accident_Year,int64,94406,0,0.00,7
7,Accident_Date,str,94406,0,0.00,2557
8,Location,str,92330,2076,2.20,14900
9,Classification_Of_Accident,str,94406,0,0.00,4


In [17]:
num_cols = [c for c in df.columns if c.startswith("num_of_")]

rows = []
for col in num_cols:
    null_mask = df[col].isna()
    total_nulls = null_mask.sum()
    pd_only_nulls = (null_mask & (df["Classification_Of_Accident"] == "03 - P.D. only")).sum()
    injury_nulls = (null_mask & df["Classification_Of_Accident"].isin(["01 - Fatal injury", "02 - Non-fatal injury"])).sum()
    other_nulls = total_nulls - pd_only_nulls - injury_nulls

    rows.append({
        "column": col,
        "total_nulls": total_nulls,
        "null_on_PD_only": pd_only_nulls,
        "null_on_injury": injury_nulls,
        "null_on_other": other_nulls,
        "safe_to_zero_fill": injury_nulls == 0,
    })

pd.DataFrame(rows)


,column,total_nulls,null_on_PD_only,null_on_injury,null_on_other,safe_to_zero_fill
0,num_of_vehicles,168,125,8,35,False
1,num_of_pedestrians,92575,76614,13398,2563,False
2,num_of_bicycles,92741,76426,13836,2479,False
3,num_of_motorcycles,93615,76464,14608,2543,False
4,num_of_injuries,79183,76612,8,2563,False
5,num_of_minimal,87816,76670,8583,2563,False
6,num_of_minor,85750,76671,6516,2563,False
7,num_of_major,93589,76671,14355,2563,False
8,num_of_fatal,94237,76671,15003,2563,False


In [16]:
# check max_injury nulls vs classification
# Summarise as a crosstab for a cleaner view
ct = pd.crosstab(
    df["Classification_Of_Accident"],
    df["Max_injury"].isna(),
    colnames=["Max_injury is null"]
)
ct.columns = ["Max_injury present", "Max_injury null"]
print(ct)
print()

# Flag the anomalies explicitly
injury_with_null = df[
    df["Max_injury"].isna() &
    df["Classification_Of_Accident"].isin(["01 - Fatal injury", "02 - Non-fatal injury"])
]
print(f"Injury rows missing Max_injury: {len(injury_with_null)}")

pd_with_value = df[
    df["Max_injury"].notna() &
    (df["Classification_Of_Accident"] == "03 - P.D. only")
]
print(f"P.D. only rows with a Max_injury value: {len(pd_with_value)}")


                            Max_injury present  Max_injury null
Classification_Of_Accident                                     
01 - Fatal injury                          169                1
02 - Non-fatal injury                    14995                7
03 - P.D. only                               1            76670
04 - Non-reportable                          0             2563

Injury rows missing Max_injury: 8
P.D. only rows with a Max_injury value: 1


## Categorical Value Audit

Value counts (including nulls) for key categorical columns. These fields will later require code-prefix cleaning, but no cleaning is done here.

In [6]:
CATEGORICAL_COLS = [
    "Classification_Of_Accident",
    "Initial_Impact_Type",
    "Light",
    "Traffic_Control",
    "Road_1_Surface_Condition",
    "Environment_Condition_1",
]

for col in CATEGORICAL_COLS:
    if col not in df.columns:
        print(f"WARNING: Column '{col}' not found in dataset — skipping.\n")
        continue
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string())
    print()

--- Classification_Of_Accident ---
Classification_Of_Accident
03 - P.D. only           76671
02 - Non-fatal injury    15002
04 - Non-reportable       2563
01 - Fatal injury          170

--- Initial_Impact_Type ---
Initial_Impact_Type
03 - Rear end                  31720
07 - SMV other                 14197
04 - Sideswipe                 13616
02 - Angle                     11825
05 - Turning movement          11164
06 - SMV unattended vehicle     8206
99 - Other                      2387
01 - Approaching                1279
NaN                               12

--- Light ---
Light
01 - Daylight    63755
07 - Dark        21207
05 - Dusk         4310
00 - Unknown      2714
03 - Dawn         2365
99 - Other          41
NaN                 14

--- Traffic_Control ---
Traffic_Control
10 - No control            43700
01 - Traffic signal        38250
02 - Stop sign             10185
11 - Roundabout             1309
03 - Yield sign              615
12 - IPS                     130
99 - Other 

## Count Column Null Audit

Quantify null rates in numeric count fields. No filling or imputation is done here.

In [7]:
COUNT_COLS = [
    "num_of_vehicles",
    "num_of_pedestrians",
    "num_of_bicycles",
    "num_of_motorcycles",
    "num_of_injuries",
    "num_of_minimal",
    "num_of_minor",
    "num_of_major",
    "num_of_fatal",
]

missing_cols = [c for c in COUNT_COLS if c not in df.columns]
if missing_cols:
    print(f"WARNING: These count columns were not found: {missing_cols}")

available_count_cols = [c for c in COUNT_COLS if c in df.columns]

count_audit = pd.DataFrame({
    "column": available_count_cols,
    "null_count": [df[c].isna().sum() for c in available_count_cols],
    "null_pct": [(df[c].isna().sum() / len(df) * 100).round(2) for c in available_count_cols],
    "non_null_count": [df[c].notna().sum() for c in available_count_cols],
    "min": [df[c].min() for c in available_count_cols],
    "max": [df[c].max() for c in available_count_cols],
    "mean": [df[c].mean().round(3) for c in available_count_cols],
})

count_audit

,column,null_count,null_pct,non_null_count,min,max,mean
0,num_of_vehicles,168,0.18,94238,1.0,25.0,1.866
1,num_of_pedestrians,92575,98.06,1831,1.0,3.0,1.039
2,num_of_bicycles,92741,98.24,1665,1.0,3.0,1.010
3,num_of_motorcycles,93615,99.16,791,1.0,3.0,1.015
4,num_of_injuries,79183,83.87,15223,1.0,38.0,1.293
5,num_of_minimal,87816,93.02,6590,1.0,11.0,1.208
6,num_of_minor,85750,90.83,8656,1.0,10.0,1.220
7,num_of_major,93589,99.13,817,1.0,14.0,1.113
8,num_of_fatal,94237,99.82,169,1.0,3.0,1.065


## Coordinate Validity Check

Ottawa bounding box: latitude 44.9–45.7, longitude -76.4–-75.2.

In [8]:
print("Lat/Long summary statistics:")
df[["Lat", "Long"]].describe()

Lat/Long summary statistics:


,Lat,Long
count,94406.000000,94406.000000
mean,45.334280,-75.709057
std,1.254323,0.157548
min,0.000000,-79.237290
25%,45.332566,-75.754909
50%,45.378657,-75.697095
75%,45.418357,-75.642588
max,45.524921,-75.261583


In [9]:
missing_coords = df[["Lat", "Long"]].isna().any(axis=1).sum()
print(f"Rows missing Lat or Long: {missing_coords:,} ({missing_coords / len(df) * 100:.2f}%)")

Rows missing Lat or Long: 0 (0.00%)


In [10]:
LAT_MIN, LAT_MAX = 44.9, 45.7
LON_MIN, LON_MAX = -76.4, -75.2

has_coords = df["Lat"].notna() & df["Long"].notna()

in_bbox = (
    df["Lat"].between(LAT_MIN, LAT_MAX) &
    df["Long"].between(LON_MIN, LON_MAX)
)

outliers = df[has_coords & ~in_bbox]
print(f"Coordinate outliers (outside Ottawa bounding box): {len(outliers):,}")

Coordinate outliers (outside Ottawa bounding box): 72


In [11]:
if len(outliers) > 0:
    display_cols = [c for c in ["ID", "Accident_Date", "Location", "Lat", "Long"] if c in df.columns]
    print("Sample outlier rows (up to 20):")
    outliers[display_cols].head(20)
else:
    print("No coordinate outliers found.")

Sample outlier rows (up to 20):


## Identifier Uniqueness Check

In [12]:
ID_COLS = ["ID", "ObjectId", "Geo_ID"]

for col in ID_COLS:
    if col not in df.columns:
        print(f"WARNING: Column '{col}' not found — skipping.")
        continue
    total = len(df)
    unique = df[col].nunique()
    nulls = df[col].isna().sum()
    is_unique = unique == total - nulls
    print(f"{col}: {unique:,} unique / {total:,} rows / {nulls:,} nulls — {'UNIQUE' if is_unique else 'NOT UNIQUE'}")

ID: 94,406 unique / 94,406 rows / 0 nulls — UNIQUE
ObjectId: 94,406 unique / 94,406 rows / 0 nulls — UNIQUE
Geo_ID: 14,900 unique / 94,406 rows / 2,076 nulls — NOT UNIQUE


## Initial Findings for Schema Documentation

- **Total rows:** 94,406
- **Total columns:** 28
- **Candidate unique row identifier:** `ID` or `ObjectId` — both are fully unique with 0 nulls. `Geo_ID` is NOT a row identifier; it represents intersection/location codes (14,900 unique values across 94,406 rows).
- **Coordinate quality summary:** No missing Lat/Long values. However, 72 rows fall outside the Ottawa bounding box (lat 44.9–45.7, lon -76.4–-75.2). The minimum Lat value is 0.0, indicating at least some rows have placeholder/corrupt coordinates. The dataset contains three coordinate representations: `Lat`/`Long` (WGS84), `X`/`Y` (Web Mercator), and `X_Coordinate`/`Y_Coordinate` (projected). Use `Lat`/`Long` for geospatial work.
- **Count column null pattern:** `num_of_vehicles` is nearly complete (0.18% null). `num_of_pedestrians`, `num_of_bicycles`, and `num_of_motorcycles` are ~98–99% null — sparse columns populated only when that road user type was involved. Injury breakdown columns (`num_of_injuries`, `num_of_minimal`, `num_of_minor`, `num_of_major`, `num_of_fatal`) are 84–99.8% null — nulls here indicate property-damage-only collisions, not missing data.
- **Categorical fields requiring code-prefix cleaning:** All 6 audited fields use `NN - Description` format and will need `strip_code_prefix()` applied before analysis: `Classification_Of_Accident`, `Initial_Impact_Type`, `Light`, `Traffic_Control`, `Road_1_Surface_Condition`, `Environment_Condition_1`.
- **Notes to transfer into `docs/data_dictionary.md`:**
  - `Max_injury` is 83.94% null. Cross-checking against `Classification_Of_Accident` confirms that nulls are overwhelmingly property-damage-only rows (76,670 P.D. only + 2,563 non-reportable). However, 8 genuine injury rows (7 non-fatal, 1 fatal) are missing `Max_injury` — these are real data quality gaps. One `P.D. only` row has a non-null `Max_injury`, likely a data entry error.
  - `Location` and `Geo_ID` share the same 2,076 null rows (2.20%) — likely collisions with no fixed intersection reference.
  - `Accident_Date` is stored as a string (`M/D/YYYY`) — will need parsing on ingest.
  - `Accident_Year` spans 7 distinct years (int64, no nulls).
  - 170 fatal collisions in the dataset (`01 - Fatal injury` in `Classification_Of_Accident`).
  - 72 coordinate outliers to investigate or flag before geospatial analysis.